### This the code which will implement advanced feature engineering that includes previous order history, purchase and return history of an customer i.e adding more historical features to existing features

## 1. Environment Setup & Library Imports


In [1]:
import pandas as pd
import numpy as np
import time
import os
import joblib
import gc
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve
)

print(f"LightGBM Version: {lgb.__version__}")
print(f"Pandas Version:   {pd.__version__}")
print(f"NumPy Version:    {np.__version__}")

LightGBM Version: 4.7.0
Pandas Version:   2.2.3
NumPy Version:    2.2.3


## 2. Data Ingestion


In [2]:
candidate_paths = [
    "cleaned_ecommerce_orders.parquet",
    "cleaned_ecommerce_dataset.parquet",
    os.path.join("notebooks", "cleaned_ecommerce_orders.parquet"),
    os.path.join("..", "notebooks", "cleaned_ecommerce_orders.parquet")
]

dataset_path = None
for p in candidate_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError("Could not find cleaned parquet dataset in any expected directory.")

t0 = time.time()
print(f"Loading cleaned dataset from: {dataset_path} ...")
df = pd.read_parquet(dataset_path)
print(f"Loaded {df.shape[0]:,} records and {df.shape[1]} columns in {time.time()-t0:.2f} seconds.")

display(df.head(5))


Loading cleaned dataset from: cleaned_ecommerce_orders.parquet ...
Loaded 3,999,025 records and 13 columns in 8.63 seconds.


,order_id,customer_id,order_date,product_category,product_id,quantity,unit_price,order_total,payment_method,shipping_cost,order_status,discount_applied,return_target
0,ORD0000000000,CUST06438651,2024-12-03,grocery,SKU-GRO-34044,5,4.83,31.59,credit_card,7.44,delivered,False,0
1,ORD0000000001,CUST04024143,2025-03-19,home,SKU-HOM-22793,1,101.25,101.25,credit_card,0.00,delivered,False,0
2,ORD0000000003,CUST05454291,2025-07-27,electronics,SKU-ELE-39207,4,416.63,1666.52,credit_card,0.00,delivered,False,0
3,ORD0000000004,CUST04434141,2024-03-11,home,SKU-HOM-09643,3,104.94,314.82,paypal,0.00,delivered,False,0
4,ORD0000000005,CUST04504595,2025-02-02,clothing,SKU-CLO-19336,1,40.34,44.76,credit_card,4.42,delivered,False,0


## 3. Constructing history based features to capture past orders data (by customers)

In [3]:
# ==============================================================================
# BASE FEATURE ENGINEERING & MEMORY DOWNCASTING
# ==============================================================================
# Sort strictly chronologically by order_date and order_id
df = df.sort_values(['order_date', 'order_id']).reset_index(drop=True)

# 1. Authentic Chronological Customer History (Strictly Past Orders < t)
# cumcount() gives 0 for 1st order, 1 for 2nd order, etc.
df['customer_previous_orders'] = df.groupby('customer_id').cumcount().astype('int16')

# Cumulative sum of returns INCLUDING current row, minus current row's target = STRICTLY PAST returns!
df['customer_return_count'] = (
    df.groupby('customer_id')['return_target'].cumsum() - df['return_target']
).astype('int16')

# Smoothed historical customer return rate (defaults to global prior 0.113 for 1st-time buyers)
global_prior = 0.1130
df['customer_return_rate'] = (
    (df['customer_return_count'] + 5.0 * global_prior) / (df['customer_previous_orders'] + 5.0)
).astype('float32')

# Customer Historical Average Spend (Strictly Past Orders < t)
past_spend_sum = df.groupby('customer_id')['unit_price'].cumsum() - df['unit_price']
df['customer_avg_purchase_price'] = np.where(
    df['customer_previous_orders'] > 0,
    past_spend_sum / df['customer_previous_orders'],
    df['unit_price'] # If 1st order, ratio is 1.0
).astype('float32')

df['splurge_price_ratio'] = (df['unit_price'] / (df['customer_avg_purchase_price'] + 1e-5)).astype('float32')

# 2. Authentic Chronological Product (SKU) History (Strictly Past Orders < t)
df['product_previous_orders'] = df.groupby('product_id').cumcount().astype('int16')
df['product_past_returns'] = (
    df.groupby('product_id')['return_target'].cumsum() - df['return_target']
).astype('int16')

# Smooth SKU return rate towards its Category's historical return rate
cat_past_orders = df.groupby('product_category', observed=True).cumcount()
cat_past_returns = df.groupby('product_category', observed=True)['return_target'].cumsum() - df['return_target']
df['category_historical_return_rate'] = (
    (cat_past_returns + 100.0 * global_prior) / (cat_past_orders + 100.0)
).astype('float32')

df['product_historical_return_rate'] = (
    (df['product_past_returns'] + 15.0 * df['category_historical_return_rate']) /
    (df['product_previous_orders'] + 15.0)
).astype('float32')

## observing transformed df with addition of new columns:
df.head()

,order_id,customer_id,order_date,product_category,product_id,quantity,unit_price,order_total,payment_method,shipping_cost,...,return_target,customer_previous_orders,customer_return_count,customer_return_rate,customer_avg_purchase_price,splurge_price_ratio,product_previous_orders,product_past_returns,category_historical_return_rate,product_historical_return_rate
0,ORD0000000982,CUST05036329,2024-01-01,grocery,SKU-GRO-12255,1,11.71,16.61,credit_card,4.90,...,0,0,0,0.113,11.710000,0.999999,0,0,0.113000,0.113000
1,ORD0000001728,CUST06266588,2024-01-01,clothing,SKU-CLO-08575,1,33.87,42.46,bnpl,8.59,...,0,0,0,0.113,33.869999,1.000000,0,0,0.113000,0.113000
2,ORD0000003068,CUST01746872,2024-01-01,clothing,SKU-CLO-40075,1,27.82,36.16,paypal,8.34,...,0,0,0,0.113,27.820000,1.000000,0,0,0.111881,0.111881
3,ORD0000004214,CUST00328099,2024-01-01,clothing,SKU-CLO-43742,2,50.29,100.58,debit_card,0.00,...,0,0,0,0.113,50.290001,1.000000,0,0,0.110784,0.110784
4,ORD0000004962,CUST05522531,2024-01-01,beauty,SKU-BEA-48540,1,10.93,20.15,credit_card,9.22,...,0,0,0,0.113,10.930000,0.999999,0,0,0.113000,0.113000


## 4. FULL Training pipeline

In [4]:
t0 = time.time()

# ==============================================================================
# 1. BASE DOMAIN FEATURES & MEMORY DOWNCASTING
# ==============================================================================
df['discount_applied'] = df['discount_applied'].astype('int8')
df['quantity'] = df['quantity'].astype('int16')
df['unit_price'] = df['unit_price'].astype('float32')
df['order_total'] = df['order_total'].astype('float32')
df['shipping_cost'] = df['shipping_cost'].astype('float32')

df['expected_subtotal'] = (df['quantity'] * df['unit_price']).astype('float32')
df['shipping_ratio'] = (df['shipping_cost'] / (df['order_total'] + 1e-5)).astype('float32')
df['effective_unit_price'] = (df['order_total'] / df['quantity']).astype('float32')
df['is_free_shipping'] = (df['shipping_cost'] == 0.0).astype('int8')
df['log_unit_price'] = np.log1p(df['unit_price']).astype('float32')
df['log_order_total'] = np.log1p(df['order_total']).astype('float32')
df['dist_from_peak_price'] = np.abs(df['unit_price'] - 70.0).astype('float32')

# Select all pure domain + authentic chronological history features
feature_cols = [
    'product_category',
    'unit_price',
    'quantity',
    'order_total',
    'shipping_cost',
    'discount_applied',
    'expected_subtotal',
    'shipping_ratio',
    'effective_unit_price',
    'is_free_shipping',
    'log_unit_price',
    'log_order_total',
    'dist_from_peak_price',
    'customer_previous_orders',
    'customer_return_count',
    'customer_return_rate',
    'customer_avg_purchase_price',
    'splurge_price_ratio',
    'product_previous_orders',
    'product_past_returns',
    'category_historical_return_rate',
    'product_historical_return_rate'
]

df['product_category'] = df['product_category'].astype('category')

# ==============================================================================
# 2. STRICT CHRONOLOGICAL SPLIT (70% Train / 15% Val / 15% Test)
# ==============================================================================
# Since df is already sorted by ['order_date', 'order_id'], we slice by time!
n_total = len(df)
train_end = int(n_total * 0.70)
val_end   = int(n_total * 0.85)

X_train = df[feature_cols].iloc[:train_end].copy()
y_train = df['return_target'].iloc[:train_end].astype('int8').copy()

X_val   = df[feature_cols].iloc[train_end:val_end].copy()
y_val   = df['return_target'].iloc[train_end:val_end].astype('int8').copy()

X_test  = df[feature_cols].iloc[val_end:].copy()
y_test  = df['return_target'].iloc[val_end:].astype('int8').copy()

# Compute Intra-Category Price Statistics strictly on X_train
cat_stats = X_train.groupby('product_category', observed=True)['unit_price'].agg(['mean', 'std', 'median']).to_dict()

for split_df in [X_train, X_val, X_test]:
    c_mean = split_df['product_category'].map(cat_stats['mean']).astype('float32')
    c_std  = split_df['product_category'].map(cat_stats['std']).astype('float32')
    c_med  = split_df['product_category'].map(cat_stats['median']).astype('float32')

    split_df['price_to_cat_mean'] = (split_df['unit_price'] / (c_mean + 1e-5)).astype('float32')
    split_df['price_cat_zscore']  = ((split_df['unit_price'] - c_mean) / (c_std + 1e-5)).astype('float32')
    split_df['price_diff_cat_median'] = (split_df['unit_price'] - c_med).astype('float32')

print(f"Chronological Split Completed in {time.time() - t0:.2f}s | Total Features: {X_train.shape[1]}")
print(f"  Train Rows: {len(X_train):,} (Return Rate: {y_train.mean():.2%})")
print(f"  Val Rows:   {len(X_val):,} (Return Rate: {y_val.mean():.2%})")
print(f"  Test Rows:  {len(X_test):,} (Return Rate: {y_test.mean():.2%})\n")

# Diagnostic: Check how much repeat history accumulated by the Test period!
print("Test Set History Richness:")
print(f"  Repeat Customer Orders (prev_orders > 0): {(X_test['customer_previous_orders'] > 0).mean():.2%}")
print(f"  Avg Previous Orders per SKU in Test Set : {X_test['product_previous_orders'].mean():.1f} orders\n")

# ==============================================================================
# 3. TRAIN CALIBRATED LIGHTGBM MODEL (NO scale_pos_weight)
# ==============================================================================
train_ds = lgb.Dataset(X_train, label=y_train, categorical_feature=['product_category'], free_raw_data=False)
val_ds   = lgb.Dataset(X_val, label=y_val, reference=train_ds, categorical_feature=['product_category'], free_raw_data=False)

params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': ['binary_logloss', 'auc'],
    'first_metric_only': True,        # Early stop on calibrated log-loss
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 1000,        # High support prevents overfitting on noisy SKUs
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 10.0,
    'reg_lambda': 25.0,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("Training LightGBM on Authentic Chronological History...")
model = lgb.train(
    params=params,
    train_set=train_ds,
    num_boost_round=400,
    valid_sets=[train_ds, val_ds],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=True),
        lgb.log_evaluation(period=25)
    ]
)

# ==============================================================================
# 4. EVALUATE TEST SET METRICS & FEATURE IMPORTANCE
# ==============================================================================
test_probs = model.predict(X_test, num_iteration=model.best_iteration)

print("\n========================================================")
print(f"Authentic Chronological Test ROC-AUC : {roc_auc_score(y_test, test_probs):.5f}")
print(f"Authentic Chronological Test PR-AUC  : {average_precision_score(y_test, test_probs):.5f}")
print(f"Calibrated Probability Range         : [{test_probs.min():.4f}, {test_probs.max():.4f}]")
print("========================================================\n")

# Feature Importance Check
imp_df = pd.DataFrame({
    'Feature': model.feature_name(),
    'Gain': model.feature_importance(importance_type='gain'),
    'Split': model.feature_importance(importance_type='split')
}).sort_values('Gain', ascending=False)

print("Top 12 Features by Information Gain:")
print(imp_df.head(12).to_string(index=False))

# Optimal F1 & F2 Thresholds
prec, rec, thresh = precision_recall_curve(y_test, test_probs)
f1_arr = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-9)
f2_arr = (5 * prec[:-1] * rec[:-1]) / (4 * prec[:-1] + rec[:-1] + 1e-9)

best_f1_idx = np.argmax(f1_arr)
best_f2_idx = np.argmax(f2_arr)

print(f"\n1. Balanced F1 Optimal Cutoff (tau = {thresh[best_f1_idx]:.4f}):")
print(f"   -> F1 = {f1_arr[best_f1_idx]:.4f} | Precision = {prec[best_f1_idx]:.2%} | Recall = {rec[best_f1_idx]:.2%}")

print(f"\n2. Recall-Priority F2 Cutoff (tau = {thresh[best_f2_idx]:.4f}):")
print(f"   -> F2 = {f2_arr[best_f2_idx]:.4f} | Precision = {prec[best_f2_idx]:.2%} | Recall = {rec[best_f2_idx]:.2%}")

Chronological Split Completed in 4.47s | Total Features: 25
  Train Rows: 2,799,317 (Return Rate: 11.30%)
  Val Rows:   599,854 (Return Rate: 11.29%)
  Test Rows:  599,854 (Return Rate: 11.29%)

Test Set History Richness:
  Repeat Customer Orders (prev_orders > 0): 30.90%
  Avg Previous Orders per SKU in Test Set : 10.4 orders

Training LightGBM on Authentic Chronological History...
Training until validation scores don't improve for 30 rounds
[25]	train's binary_logloss: 0.332315	train's auc: 0.685794	val's binary_logloss: 0.331874	val's auc: 0.686054
[50]	train's binary_logloss: 0.330213	train's auc: 0.686766	val's binary_logloss: 0.329795	val's auc: 0.686289
Early stopping, best iteration is:
[39]	train's binary_logloss: 0.330641	train's auc: 0.686397	val's binary_logloss: 0.330205	val's auc: 0.686502

Authentic Chronological Test ROC-AUC : 0.68467
Authentic Chronological Test PR-AUC  : 0.19062
Calibrated Probability Range         : [0.0330, 0.2076]

Top 12 Features by Information Ga

## 5. Saving Model and its metadata inside artifacts folder

In [5]:
# Ensure artifacts directory exists
artifacts_dir = os.path.join("..", "artifacts", "models")
if not os.path.exists(artifacts_dir):
    artifacts_dir = "artifacts/models"
os.makedirs(artifacts_dir, exist_ok=True)

# 1. Save LightGBM booster
txt_model_path = os.path.join(artifacts_dir, "lightgbm_model_with_historical_features.txt")
joblib_model_path = os.path.join(artifacts_dir, "lightgbm_model_with_historical_features.joblib")

model.save_model(txt_model_path)
joblib.dump(model, joblib_model_path)
print(f"\nModel saved to:\n  -> {txt_model_path}\n  -> {joblib_model_path}")




Model saved to:
  -> ..\artifacts\models\lightgbm_model_with_historical_features.txt
  -> ..\artifacts\models\lightgbm_model_with_historical_features.joblib


In [6]:
df[df['customer_return_count'] >= 1]

,order_id,customer_id,order_date,product_category,product_id,quantity,unit_price,order_total,payment_method,shipping_cost,...,product_past_returns,category_historical_return_rate,product_historical_return_rate,expected_subtotal,shipping_ratio,effective_unit_price,is_free_shipping,log_unit_price,log_order_total,dist_from_peak_price
14636,ORD0003174891,CUST04326907,2024-01-03,beauty,SKU-BEA-17413,2,16.299999,41.599998,debit_card,9.00,...,0,0.073933,0.073933,32.599998,0.216346,20.799999,0,2.850706,3.751854,53.700001
18544,ORD0001676599,CUST06534962,2024-01-04,clothing,SKU-CLO-06723,2,116.860001,233.720001,paypal,0.00,...,0,0.205427,0.205427,233.720001,0.000000,116.860001,1,4.769497,5.458393,46.860001
20255,ORD0003228584,CUST03279790,2024-01-04,home,SKU-HOM-24176,4,60.549999,242.199997,credit_card,0.00,...,0,0.090459,0.090459,242.199997,0.000000,60.549999,1,4.119850,5.493884,9.450001
24646,ORD0002262602,CUST09685567,2024-01-05,toys,SKU-TOY-17866,1,56.320000,56.320000,credit_card,0.00,...,0,0.087634,0.087634,56.320000,0.000000,56.320000,1,4.048650,4.048650,13.680000
27817,ORD0000118852,CUST08615854,2024-01-06,electronics,SKU-ELE-19844,1,427.820007,427.820007,bnpl,0.00,...,0,0.141486,0.141486,427.820007,0.000000,427.820007,1,6.061037,6.061037,357.820007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3998813,ORD0004810582,CUST00433459,2025-12-30,electronics,SKU-ELE-33778,1,701.869995,701.869995,debit_card,0.00,...,3,0.139059,0.175376,701.869995,0.000000,701.869995,1,6.555172,6.555172,631.869995
3998825,ORD0004821555,CUST09496148,2025-12-30,clothing,SKU-CLO-37163,1,22.809999,31.129999,credit_card,8.32,...,0,0.220157,0.143581,22.809999,0.267266,31.129999,0,3.170106,3.469790,47.190002
3998857,ORD0004846409,CUST06552208,2025-12-30,sports,SKU-SPO-01087,1,128.720001,128.720001,paypal,0.00,...,1,0.110533,0.115565,128.720001,0.000000,128.720001,1,4.865378,4.865378,58.720001
3998928,ORD0004906454,CUST05841762,2025-12-30,home,SKU-HOM-03578,2,139.610001,279.220001,credit_card,0.00,...,1,0.089262,0.111378,279.220001,0.000000,139.610001,1,4.945990,5.635575,69.610001


In [7]:
df[df['customer_previous_orders'] >= 1]

,order_id,customer_id,order_date,product_category,product_id,quantity,unit_price,order_total,payment_method,shipping_cost,...,product_past_returns,category_historical_return_rate,product_historical_return_rate,expected_subtotal,shipping_ratio,effective_unit_price,is_free_shipping,log_unit_price,log_order_total,dist_from_peak_price
3979,ORD0003583511,CUST00473919,2024-01-01,home,SKU-HOM-22762,1,85.379997,85.379997,paypal,0.00,...,0,0.085234,0.085234,85.379997,0.000000,85.379997,1,4.458756,4.458756,15.379997
4356,ORD0003897968,CUST01345183,2024-01-01,toys,SKU-TOY-06776,1,66.459999,66.459999,credit_card,0.00,...,0,0.097343,0.097343,66.459999,0.000000,66.459999,1,4.211535,4.211535,3.540001
4455,ORD0003981091,CUST06164697,2024-01-01,electronics,SKU-ELE-23753,2,263.980011,527.960022,credit_card,0.00,...,0,0.147294,0.147294,527.960022,0.000000,263.980011,1,5.579654,6.270913,193.980011
4550,ORD0004069430,CUST01680032,2024-01-01,sports,SKU-SPO-26661,4,64.480003,257.920013,debit_card,0.00,...,0,0.108113,0.108113,257.920013,0.000000,64.480003,1,4.181745,5.556519,5.519997
6439,ORD0000799943,CUST00201536,2024-01-02,sports,SKU-SPO-48088,1,36.509998,42.779999,debit_card,6.27,...,0,0.105690,0.105690,36.509998,0.146564,42.779999,0,3.624608,3.779177,33.490002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3999010,ORD0004988640,CUST03516807,2025-12-30,beauty,SKU-BEA-22166,2,43.910000,87.820000,debit_card,0.00,...,1,0.070322,0.076105,87.820000,0.000000,43.910000,1,3.804661,4.486612,26.090000
3999013,ORD0004989584,CUST08881702,2025-12-30,grocery,SKU-GRO-30892,2,16.510000,37.770000,debit_card,4.75,...,1,0.020169,0.065127,33.020000,0.125761,18.885000,0,2.862772,3.657647,53.489998
3999020,ORD0004995365,CUST09958886,2025-12-30,clothing,SKU-CLO-02508,1,65.779999,65.779999,credit_card,0.00,...,5,0.220156,0.244186,65.779999,0.000000,65.779999,1,4.201404,4.201404,4.220001
3999023,ORD0004998875,CUST05989195,2025-12-30,sports,SKU-SPO-17626,2,143.660004,287.320007,credit_card,0.00,...,0,0.110534,0.066320,287.320007,0.000000,143.660004,1,4.974386,5.664071,73.660004


### Above code run shows that only there are 88689 instances of an previous returns by an customer 39,10,336 customers has previous returns which is our main point of interest , also there are total 702597 customers who have ordered again so in total they only conists of 17% of repeated customers so here by calculation we hold only historical data of 17% of all customers in our platform which atleast had done 1 purchase before (non new buyer categorry)

### Due to mathematical constraints explained in ecommerce_data_mathematical_ceiling_analysis.md file we cannot directly use this model to output raw labels "delivered" and "returned" based on input features instead to make models more reliable for predictions we will going to introduce confidence scores and helps to give confidence tiers, alarm triggered or not and their recommended actions for level of predicted risk so for inference we are going to use confidence score driven inference for outputting the predicted return probability based on distribution of returns observed in this dataset

In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb


class ECommerceReturnPredictor:
    def __init__(self, model_path, baseline_rate=0.1130):
        """
        Production inference engine for native LightGBM Booster saved via joblib.
        Auto-extracts feature names and handles cold-start customer/product history.
        """
        self.baseline_rate = baseline_rate
        
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found at: {model_path}")

        # 1. Load the model from .joblib or .txt
        if model_path.endswith('.txt'):
            self.model = lgb.Booster(model_file=model_path)
        else:
            self.model = joblib.load(model_path)

        # 2. Extract best_iteration
        self.best_iteration = getattr(self.model, 'best_iteration', None)
        if self.best_iteration is None or self.best_iteration <= 0:
            self.best_iteration = self.model.num_trees()

        # 3. Extract exact expected feature names from the Booster
        self.feature_names = self.model.feature_name()

        # 4. Category return rate priors (from EDA)
        self.cat_rates = {
            'clothing': 0.2219, 'electronics': 0.1375, 'sports': 0.1103,
            'home': 0.0874, 'toys': 0.0806, 'beauty': 0.0732,
            'books': 0.0396, 'grocery': 0.0197
        }

        # 5. Category price benchmark stats
        self.cat_price_mean = {
            'clothing': 128.45, 'electronics': 353.72, 'sports': 89.20,
            'home': 114.30, 'toys': 47.10, 'beauty': 38.60,
            'books': 22.40, 'grocery': 12.85
        }
        self.cat_price_std = {
            'clothing': 78.10, 'electronics': 189.50, 'sports': 52.30,
            'home': 65.40, 'toys': 31.20, 'beauty': 24.10,
            'books': 14.80, 'grocery': 8.90
        }
        self.cat_price_med = {
            'clothing': 65.50, 'electronics': 240.00, 'sports': 72.00,
            'home': 95.00, 'toys': 38.00, 'beauty': 31.00,
            'books': 18.00, 'grocery': 9.80
        }

    def transform_features(self, raw_df):
        """
        Engineers base domain features and safely imputes cold-start
        history columns if they are missing from raw checkout payloads.
        """
        df = raw_df.copy()

        # Date handling
        if 'order_date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['order_date']):
            df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

        # Typecast core attributes
        df['discount_applied'] = df['discount_applied'].astype('int8')
        df['quantity'] = df['quantity'].astype('int16')
        df['unit_price'] = df['unit_price'].astype('float32')
        df['order_total'] = df['order_total'].astype('float32')
        df['shipping_cost'] = df['shipping_cost'].astype('float32')

        # 1. Base Domain & Accounting Features
        df['expected_subtotal'] = (df['quantity'] * df['unit_price']).astype('float32')
        df['shipping_ratio'] = (df['shipping_cost'] / (df['order_total'] + 1e-5)).astype('float32')
        df['effective_unit_price'] = (df['order_total'] / df['quantity']).astype('float32')
        df['is_free_shipping'] = (df['shipping_cost'] == 0.0).astype('int8')

        # 2. Non-linear Price Transformations
        df['log_unit_price'] = np.log1p(df['unit_price']).astype('float32')
        df['log_order_total'] = np.log1p(df['order_total']).astype('float32')
        df['dist_from_peak_price'] = np.abs(df['unit_price'] - 70.0).astype('float32')

        # 3. Category Relative Price Features
        c_mean = df['product_category'].astype(str).map(self.cat_price_mean).fillna(df['unit_price'].mean()).astype('float32')
        c_std  = df['product_category'].astype(str).map(self.cat_price_std).fillna(df['unit_price'].std()).astype('float32')
        c_med  = df['product_category'].astype(str).map(self.cat_price_med).fillna(df['unit_price'].median()).astype('float32')

        df['price_to_cat_mean'] = (df['unit_price'] / (c_mean + 1e-5)).astype('float32')
        df['price_cat_zscore']  = ((df['unit_price'] - c_mean) / (c_std + 1e-5)).astype('float32')
        df['price_diff_cat_median'] = (df['unit_price'] - c_med).astype('float32')

        # 4. Category Prior
        if 'category_historical_return_rate' not in df.columns:
            df['category_historical_return_rate'] = df['product_category'].astype(str).map(self.cat_rates).fillna(0.1130).astype('float32')

        # 5. Cold-Start Fallbacks for Historical Features (if missing from raw checkout)
        if 'customer_previous_orders' not in df.columns:
            df['customer_previous_orders'] = np.int16(0)
        if 'customer_return_count' not in df.columns:
            df['customer_return_count'] = np.int16(0)
        if 'customer_return_rate' not in df.columns:
            df['customer_return_rate'] = np.float32(0.1130)
        if 'customer_avg_purchase_price' not in df.columns:
            df['customer_avg_purchase_price'] = df['unit_price'].astype('float32')
        if 'splurge_price_ratio' not in df.columns:
            df['splurge_price_ratio'] = (df['unit_price'] / (df['customer_avg_purchase_price'] + 1e-5)).astype('float32')
        if 'product_previous_orders' not in df.columns:
            df['product_previous_orders'] = np.int16(0)
        if 'product_past_returns' not in df.columns:
            df['product_past_returns'] = np.int16(0)
        if 'product_historical_return_rate' not in df.columns:
            df['product_historical_return_rate'] = df['category_historical_return_rate']

        # Native LightGBM categorical encoding
        df['product_category'] = df['product_category'].astype('category')
        if 'payment_method' in df.columns:
            df['payment_method'] = df['payment_method'].astype('category')

        # Align columns to match Booster feature names
        return df[self.feature_names]

    def predict(self, raw_input_df):
        """
        Executes prediction on Booster and returns formatted decisions and action tiers.
        """
        X_proc = self.transform_features(raw_input_df)
        
        # Native Booster .predict() returns probabilities directly
        raw_probs = self.model.predict(X_proc, num_iteration=self.best_iteration)

        results = []
        for p in raw_probs:
            rel_risk = p / self.baseline_rate

            if p >= 0.20:
                tier = "HIGH_CONFIDENCE_RETURN"
                alert = True
                action = "RESTRICT_COD_OR_REQUIRE_PREPAYMENT"
            elif p >= 0.14:
                tier = "MODERATE_RISK"
                alert = False
                action = "TRIGGER_SIZING_MODAL_OR_CARE_EMAIL"
            elif p >= 0.08:
                tier = "NEUTRAL"
                alert = False
                action = "STANDARD_DISPATCH"
            else:
                tier = "HIGH_CONFIDENCE_DELIVERED"
                alert = False
                action = "PRIORITY_1_CLICK_FAST_TRACK"

            results.append({
                "predicted_return_prob": round(float(p), 4),
                "relative_risk_lift": f"{rel_risk:.2f}x",
                "confidence_tier": tier,
                "alarm_triggered": alert,
                "recommended_action": action
            })

        output_df = pd.DataFrame(results)
        return pd.concat([raw_input_df.reset_index(drop=True), output_df], axis=1)

In [10]:

# 1. Providing path and loading the saved model from artifacts directory
model_path = "../artifacts/models/lightgbm_model_with_historical_features.joblib"

model = joblib.load(model_path)

print("Model loaded successfully!")


# 2. Instantiate predictor
predictor = ECommerceReturnPredictor(model_path=model_path)

# 3. Test on a raw incoming order batch (no history columns needed)
sample_orders = pd.DataFrame([
    {
        'order_id': 'TEST_001',
        'product_category': 'clothing',
        'unit_price': 85.00,
        'quantity': 1,
        'order_total': 85.00,
        'shipping_cost': 0.00,
        'discount_applied': 0,
        'order_date': '2026-11-10'
    },
    {
        'order_id': 'TEST_002',
        'product_category': 'grocery',
        'unit_price': 8.50,
        'quantity': 3,
        'order_total': 32.50,
        'shipping_cost': 7.00,
        'discount_applied': 1,
        'order_date': '2026-11-10'
    },
    {
        'order_id': 'ORD_NEW_003',
        'product_category': 'electronics',
        'unit_price': 420.00,
        'quantity': 1,
        'order_total': 420.00,
        'shipping_cost': 0.00,
        'discount_applied': 0,
        'order_date': '2026-11-10'
    }
])

# 4. Predict
predictions = predictor.predict(sample_orders)

cols = [
    'order_id', 'product_category', 'unit_price',
    'predicted_return_prob', 'relative_risk_lift',
    'confidence_tier', 'alarm_triggered', 'recommended_action'
]
print(predictions[cols].to_string(index=False))

Model loaded successfully!
   order_id product_category  unit_price  predicted_return_prob relative_risk_lift           confidence_tier  alarm_triggered                 recommended_action
   TEST_001         clothing        85.0                 0.2065              1.83x    HIGH_CONFIDENCE_RETURN             True RESTRICT_COD_OR_REQUIRE_PREPAYMENT
   TEST_002          grocery         8.5                 0.0330              0.29x HIGH_CONFIDENCE_DELIVERED            False        PRIORITY_1_CLICK_FAST_TRACK
ORD_NEW_003      electronics       420.0                 0.1355              1.20x                   NEUTRAL            False                  STANDARD_DISPATCH
